# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này hỗ trợ cả 2 chế độ:
1. **Chạy ngầm (Background Runner - GẬP MÁY ĐI NGỦ ĐƯỢC):** Chạy độc lập trên GPU cloud của Colab bằng `nohup`. Bạn có thể gập máy tính, tắt VS Code, GPU trên mây vẫn chạy liên tục cho đến khi xong đủ 300 mẫu.
2. **Chạy trực tiếp (Foreground) với Auto-Resume:** Nhìn thấy từng mẫu chạy trên màn hình. Nếu mất mạng hoặc laptop ngủ, chạy lại sẽ tự động tiếp tục (resume) từ mẫu dở dang.

In [13]:
# 0. Kết nối Google Drive (Bắt buộc để nạp dữ liệu data.zip và khôi phục checkpoint)
from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã kết nối Google Drive thành công!')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Đã kết nối Google Drive thành công!


In [14]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

Tue Nov 10 02:05:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
# 2. Cài đặt thư viện vLLM (Chỉ cài đặt khi chưa có, tránh cài đè gây treo máy và xung đột dependency)
try:
    import vllm
    print(f'✅ vLLM đã có sẵn trong môi trường (phiên bản {vllm.__version__})! Bỏ qua bước cài đặt.')
except ImportError:
    print('⏳ Đang cài đặt vLLM...')
    !pip install -q vllm
    import vllm
    print(f'✅ Đã cài đặt thành công vLLM {vllm.__version__}!')


✅ vLLM đã có sẵn trong môi trường (phiên bản 0.33.0)! Bỏ qua bước cài đặt.


In [16]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os

# Luôn luôn xuất phát từ thư mục gốc /content (chống lỗi clone lồng nhau ZAIC_2026/ZAIC_2026)
%cd /content

if not os.path.exists('/content/ZAIC_2026/.git'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git /content/ZAIC_2026

%cd /content/ZAIC_2026
!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

/content
/content/ZAIC_2026
Already on 'planner/vuong'
Your branch is up to date with 'origin/planner/vuong'.
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 18 (delta 9), reused 18 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 188.71 KiB | 3.25 MiB/s, done.
From https://github.com/Bamboohoccode/ZAIC_2026
 * branch            planner/vuong -> FETCH_HEAD
   a1aa1aa..39db72c  planner/vuong -> origin/planner/vuong
Updating a1aa1aa..39db72c
Fast-forward
 notebooks/colab_run_planner.ipynb                  |  180 +-
 optimize/gepa/vuong/eval_results/dev_v10.jsonl     |  300 ++
 optimize/gepa/vuong/eval_results/dev_v10.jsonl.log | 4355 ++++++++++++++++++++
 .../vuong/eval_results/dev_v10.jsonl.trace.jsonl   |  404 ++
 prompts/gepa/vuong/manifest.json                   |   13 +-
 prompts/gepa/vuong/planner_v11.txt                 |   34 +
 task_flow/vuong.md                   

In [17]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Tự động mount Drive và giải nén lên ổ cứng cục bộ /content)
import os, shutil, sys
from pathlib import Path

# 4.1. Tự động Mount Google Drive nếu chưa mount
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        print('🔗 Đang kết nối Google Drive...')
        drive.mount('/content/drive')
        print('✅ Đã kết nối Google Drive thành công!')
    except Exception as e:
        print(f'⚠️ Không thể kết nối Google Drive: {e}')

%cd /content/ZAIC_2026

# 4.2. Kiểm tra xem dữ liệu zaic_tools đã có sẵn chưa
has_local_data = os.path.exists('/content/data/train/train.jsonl') and os.path.exists('/content/data/train/zaic_tools/apps/banking.py')

if not has_local_data:
    # Tìm file zip data.zip trên Drive hoặc /content
    zip_candidates = [
        '/content/drive/MyDrive/data.zip',
        '/content/drive/MyDrive/ZAIC_2026/data.zip',
        '/content/drive/MyDrive/ZAIC_2026_runs/data.zip',
        '/content/drive/MyDrive/ZAIC/data.zip',
        '/content/data.zip',
    ]
    found_zip = next((z for z in zip_candidates if os.path.exists(z)), None)
    if not found_zip and os.path.exists('/content/drive/MyDrive'):
        for f in os.listdir('/content/drive/MyDrive'):
            if f.lower().endswith('.zip') and ('data' in f.lower() or 'train' in f.lower()):
                found_zip = os.path.join('/content/drive/MyDrive', f)
                break

    if found_zip:
        print(f'📦 Đang giải nén dữ liệu từ {found_zip} sang ổ cứng cục bộ /content/ ... (mất ~15 giây)')
        # Dọn dẹp các symlink/thư mục hỏng trước khi giải nén chống lỗi 'checkdir error: File exists'
        if os.path.islink('/content/data/train') or (os.path.exists('/content/data/train') and not os.path.exists('/content/data/train/train.jsonl')):
            !rm -rf /content/data/train
        !unzip -q -o "{found_zip}" -d /content/
    elif os.path.exists('/content/drive/MyDrive/data/train'):
        print('📁 Đang sao chép thư mục data/train từ Google Drive sang /content/data/train ...')
        os.makedirs('/content/data', exist_ok=True)
        !cp -r /content/drive/MyDrive/data/train /content/data/train
    else:
        print('⚠️ Không tìm thấy data.zip trên Drive hoặc /content!')

# 4.3. Đảm bảo /content/data/train tồn tại (kể cả khi zip bung ra /content/train)
if not os.path.exists('/content/data/train') and os.path.exists('/content/train'):
    os.makedirs('/content/data', exist_ok=True)
    !mv /content/train /content/data/train

os.makedirs('data', exist_ok=True)
if os.path.exists('/content/data/train'):
    !rm -rf data/train
    !cp -r /content/data/train data/train 2>/dev/null || ln -s /content/data/train data/train
if os.path.exists('/content/data/public_test'):
    !rm -rf data/public_test
    !cp -r /content/data/public_test data/public_test 2>/dev/null || ln -s /content/data/public_test data/public_test

# 4.4. KIỂM TRA TỰ ĐỘNG: Thêm đường dẫn tuyệt đối vào sys.path để import chuẩn xác
for p in [os.path.abspath('data/train'), '/content/data/train', '/content/ZAIC_2026/data/train']:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

try:
    from zaic_tools import ToolEnv
    print('✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!')
    !ls -lh data/train | head -n 6
except Exception as e:
    print('❌ LỖI DỮ LIỆU:', e)
    print('Vui lòng kiểm tra file data.zip có chứa thư mục zaic_tools/apps hay không!')


/content/ZAIC_2026
✅ Dữ liệu data/train và module zaic_tools.apps: HOẠT ĐỘNG HOÀN HẢO!
total 7.1M
drwxr-xr-x 1547 root root  44K Nov 10 02:05 audio
drwxr-xr-x 1547 root root  44K Nov 10 02:05 train_environment
-rw-r--r--    1 root root 6.9M Nov 10 02:05 train.jsonl
-rw-r--r--    1 root root  90K Nov 10 02:05 train_tools_catalog.json
drwxr-xr-x    4 root root 4.0K Nov 10 02:05 zaic_tools


### 🎯 LỰA CHỌN CÁCH CHẠY PIPELINE:
- **Cách 5A (Khuyên dùng khi đi ngủ):** Chạy ngầm trong background với `nohup`. Bạn có thể **gập máy tính lại**, GPU Colab vẫn chạy tiếp đến khi xong!
- **Cách 5B (Khi đang ngồi máy):** Chạy trực tiếp trên màn hình có kèm cờ `--resume` (nếu lỡ tắt máy, chạy lại sẽ tự làm tiếp từ mẫu vừa dừng).

In [18]:
# 5A. [KHUYÊN DÙNG] CHẠY NGẦM BACKGROUND (GẬP MÁY / SLEEP LAPTOP THOẢI MÁI)
%cd /content/ZAIC_2026
# Dọn dẹp tiến trình cũ để giải phóng 100% VRAM GPU
!pkill -9 -f 'lifemate.planner.runner' || true
!pkill -9 -f 'lifemate.pipeline' || true
import os, time, shutil
time.sleep(2)
os.makedirs('runs', exist_ok=True)

# Tên định danh đợt chạy: dev_v12 (100% Thuần Prompt Planner V12, không dùng validator luật cứng để chống overfit)
RUN_NAME = 'dev_v12'
out_path = f'runs/{RUN_NAME}.jsonl'
log_path = f'runs/{RUN_NAME}.jsonl.log'
nohup_log = f'runs/{RUN_NAME}.nohup.log'

# Tự động khôi phục toàn bộ checkpoint, logs và trace từ Google Drive nếu máy ảo vừa được tạo mới
drive_dir = '/content/drive/MyDrive/ZAIC_2026_runs'
if os.path.exists(drive_dir):
    for fname in [f'{RUN_NAME}.jsonl', f'{RUN_NAME}.jsonl.log', f'{RUN_NAME}.jsonl.trace.jsonl']:
        src = os.path.join(drive_dir, fname)
        dst = os.path.join('runs', fname)
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)

if os.path.exists(out_path):
    with open(out_path, 'r', encoding='utf-8') as f:
        n_resumed = sum(1 for line in f if line.strip())
    print(f'🔄 Đã tìm thấy {n_resumed}/300 mẫu đã chạy trên Google Drive! Tự động nạp vào runs/ để tiếp tục từ mẫu {n_resumed + 1}.')
else:
    print('🚀 Bắt đầu chạy mới từ mẫu 1/300...')

# Chạy ngầm bằng nohup với champion prompt planner_v10 mới nhất
!nohup python -m lifemate.planner.runner \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem 0.90 \
    --max-ctx 12288 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v12.txt \
    --ids eval/splits/dev_ids.txt \
    --resume \
    --out runs/dev_v12.jsonl > runs/dev_v12.nohup.log 2>&1 &

print(f'🚀 Tiến trình {RUN_NAME} đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!')
print('Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.')


/content/ZAIC_2026
^C
^C
🚀 Bắt đầu chạy mới từ mẫu 1/300...
🚀 Tiến trình dev_v11 đã được đưa vào chạy ngầm trên GPU Cloud (chuẩn --gpu-mem 0.90 như Kaggle)!
Bạn có thể gập máy tính lại hoặc theo dõi tiến độ ở Cell 5C bên dưới.


In [19]:
# 5B. (Tuỳ chọn) CHẠY TRỰC TIẾP TRÊN MÀN HÌNH (CÓ TỰ ĐỘNG RESUME TIẾP TỤC)
# %cd /content/ZAIC_2026
# import os
# os.makedirs('runs', exist_ok=True)
#
# !python -m lifemate.planner.runner \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper none \
#     --gpu-mem 0.90 \
#     --max-ctx 12288 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v12.txt \
#     --ids eval/splits/dev_ids.txt \
#     --resume \
#     --out runs/dev_v12.jsonl


In [1]:
# 5C. THEO DÕI TIẾN ĐỘ THỜI GIAN THỰC (Tự động cập nhật liên tục)
%cd /content/ZAIC_2026
import time, os
from IPython.display import clear_output

RUN_NAME = 'dev_v12'
out_path = f'runs/{RUN_NAME}.jsonl'
log_path = f'runs/{RUN_NAME}.jsonl.log'
nohup_log = f'runs/{RUN_NAME}.nohup.log'

loop_count = 0
while True:  # Cập nhật liên tục không giới hạn 10 phút, chỉ dừng khi xong hoặc có lỗi
    clear_output(wait=True)
    loop_count += 1
    done_count = 0
    if os.path.exists(out_path):
        with open(out_path, 'r', encoding='utf-8') as f:
            done_count = sum(1 for line in f if line.strip())
    
    pids = os.popen("pgrep -f 'lifemate.planner.runner' || pgrep -f 'lifemate.pipeline'").read().strip().split()
    print(f'📊 TIẾN ĐỘ: Đã xong {done_count}/300 mẫu dev ({done_count/300*100:.1f}%)')
    print('=' * 65)
    
    # Tự động đồng bộ ngay sang Google Drive để bạn theo dõi trên điện thoại/Drive
    if os.path.exists('/content/drive/MyDrive') and done_count > 0:
        drive_dir = '/content/drive/MyDrive/ZAIC_2026_runs'
        os.makedirs(drive_dir, exist_ok=True)
        os.system(f'cp -f {out_path} {drive_dir}/ 2>/dev/null')
        os.system(f'cp -f {log_path} {drive_dir}/ 2>/dev/null')
    
    if os.path.exists(log_path) and os.path.getsize(log_path) > 0:
        !tail -n 12 {log_path}
    elif os.path.exists(nohup_log):
        print('⏳ Đang khởi động và làm nóng mô hình vLLM (Torch compile & CUDA graphs)...')
        print('Quá trình làm nóng này mất khoảng 2-3 phút trước khi chạy mẫu đầu tiên:')
        !tail -n 8 {nohup_log}
    
    if done_count >= 300:
        print('\n🎉 TIẾN TRÌNH ĐÃ HOÀN THÀNH XONG ĐỦ 300 MẪU! Hãy dừng cell này và chạy Cell 6 để xem điểm số.')
        break
    elif pids:
        print(f'\n⚡ Tiến trình GPU đang xử lý (PID: {pids[0]})... [Cập nhật #{loop_count}]')
        time.sleep(5)
    else:
        if done_count > 0:
            print(f'\n⏹️ Tiến trình đã dừng lại ở {done_count}/300 mẫu.')
            if os.path.exists(nohup_log):
                print('--- Chi tiết đuôi log nohup ---')
                !tail -n 15 {nohup_log}
            print('Có thể chạy lại Cell 5A để tự động resume tiếp tục mẫu kế tiếp.')
        else:
            print('\n⏹️ Chưa có tiến trình nào đang chạy.')
        break


📊 TIẾN ĐỘ: Đã xong 300/300 mẫu dev (100.0%)
[04:53:57]   -> Context: history_turns=22, tools=33, tokens=7015
[04:53:57]   [Step 1/8] time_left=119.9s
[04:53:57]     -> [LLM Chat] msgs=22 input_tokens=7015 calling vLLM...
[04:54:21]     -> [LLM Output in 24.0s] calls=['gold_price.get_price'] | preview: <tool_call>
[04:54:21]     -> [Tool Exec] gold_price.get_price in 0.00s -> OK
[04:54:21]   [Step 2/8] time_left=95.8s
[04:54:21]     -> [LLM Chat] msgs=24 input_tokens=7101 calling vLLM...
[04:54:32]     -> [LLM Output in 11.2s] calls=[] | preview: Giá vàng nhẫn hôm nay là 116,4 triệu VND/lượng (mua) và 118,4 triệu VND/lượng (bán).  
[04:54:32]   [Step 2] No tool calls -> Agent finished.
[04:54:32] [300/300] train_1753 DONE calls=1 35.4s
[04:54:32] 
Finished 300 samples -> runs/dev_v11.jsonl | avg 33.8s/sample | agent > 30s: 119

🎉 TIẾN TRÌNH ĐÃ HOÀN THÀNH XONG ĐỦ 300 MẪU! Hãy dừng cell này và chạy Cell 6 để xem điểm số.


In [ ]:
!nvidia-smi

Tue Nov 10 01:17:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |    8661MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# !pkill -9 -f 'lifemate.pipeline'

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
%cd /content/ZAIC_2026
import os

RUN_NAME = 'dev_v12'
sub_file = f'runs/{RUN_NAME}.jsonl'
if not os.path.exists(sub_file) or os.path.getsize(sub_file) == 0:
    print('⚠️ Chưa có dữ liệu kết quả để chấm điểm!')
    print('Lý do: Tiến trình GPU ở Cell 5 đang khởi động vLLM hoặc đang chạy những mẫu đầu tiên.')
    print('👉 Hãy đợi Cell 5C hoàn thành ít nhất 1 mẫu rồi chạy lại cell này để chấm điểm!')
else:
    with open(sub_file, 'r', encoding='utf-8') as f:
        n_samples = sum(1 for line in f if line.strip())
    print(f'Đang chấm điểm trên {n_samples}/300 mẫu đã hoàn thành...')
    !python eval/scorer.py --sub {sub_file}


/content/ZAIC_2026
Đang chấm điểm trên 300/300 mẫu đã hoàn thành...

===== KẾT QUẢ ĐÁNH GIÁ (N=300) =====
Mean Score: 0.3018
  act  : 0.3202 (n=236)
  ask  : 0.0706 (n=42)
  empty: 0.5455 (n=22)

Phân loại lỗi:
  HASH_MISMATCH       : 150 (50.0%)
  WRONG_TOOL_OR_ARGS  : 42 (14.0%)
  MISSING_CALL        : 12 (4.0%)
  FALSE_ACT           : 10 (3.3%)
  MISSED_ASK          : 10 (3.3%)
  EXTRA_CALL          : 9 (3.0%)


In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
%cd /content/ZAIC_2026
import os, shutil
from pathlib import Path

RUN_NAME = 'dev_v12'
if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in [f'{RUN_NAME}.jsonl', f'{RUN_NAME}.jsonl.log', f'{RUN_NAME}.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print(f'File kết quả hiện nằm trực tiếp tại: runs/{RUN_NAME}.jsonl')


/content/ZAIC_2026
✅ Đã lưu: dev_v10.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v10.jsonl.log -> /content/drive/MyDrive/ZAIC_2026_runs
✅ Đã lưu: dev_v10.jsonl.trace.jsonl -> /content/drive/MyDrive/ZAIC_2026_runs
Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: /content/drive/MyDrive/ZAIC_2026_runs
